# DDRI 샘플링 1안 시작 노트북

이 노트북은 `일단위 대표일 추출` 방식의 샘플링 1안을 시작하기 위한 작업 노트다.

샘플링 1안 정의:
- 정본은 유지한다.
- 샘플링은 `2023 train`에만 적용한다.
- 같은 `station_id`, 같은 `weekday` 안에서만 비교한다.
- 일단위 24시간 프로파일이 유사한 날들은 대표일 1개만 남긴다.
- 검증 `2024`, 테스트 `2025`는 축소하지 않는다.
- 임포트는 항상 별도 셀로 분리한다.


## 기준선 요약

무샘플링 기준선 1차 결과에서는 4개 조합 모두 `LightGBM_RMSE`가 우세했다.

1차 해석:
- `bike_change_deseasonalized`가 RMSE, MAE 기준으로 더 유리했다.
- 따라서 샘플링 1안은 우선 `bike_change_deseasonalized`를 기본 타깃으로 점검한다.
- 필요하면 같은 절차를 `bike_change_raw`에도 다시 적용한다.

현재 샘플링 1안 구현 메모:
- 대표일 수가 목표보다 적어지는 빈 클러스터 문제를 막기 위해 `KMeans`를 사용한다.
- 그래도 선택 수가 부족하면 거리 기준으로 대표일을 추가 보충한다.
- `retain_ratio=0.7`일 때 실제 유지율은 그룹별 `ceil` 때문에 약 `0.7123`까지 약간 올라갈 수 있다.


## 1. 임포트

이 노트북도 실행 노트북이므로 임포트만 담당하는 셀을 별도로 둔다.


In [14]:
from pathlib import Path
import importlib.util
import json
import math
import sys

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


## 2. 실행 설정

이 셀에서는 데이터셋, 타깃, 축소율 같은 실험 파라미터를 정한다.


In [15]:
ROOT = Path('/Users/cheng80/Desktop/ddri_work')

dataset_name = 'rep15'  # 'rep15' 또는 'full161'
target_col = 'bike_change_deseasonalized'  # 또는 'bike_change_raw'
retain_ratio = 0.7  # 대표일을 얼마나 남길지 결정하는 비율
min_days_per_group = 8  # 이보다 적은 그룹은 축소하지 않음

canonical_dir_map = {
    'rep15': ROOT / '3조 공유폴더' / '대표대여소_예측데이터_15개' / 'canonical_data',
    'full161': ROOT / '3조 공유폴더' / '군집별 데이터_전체 스테이션' / 'canonical_data',
}

sampled_output_dir_map = {
    'rep15': ROOT / '3조 공유폴더' / '대표대여소_예측데이터_15개' / 'sampling_plan1_outputs',
    'full161': ROOT / '3조 공유폴더' / '군집별 데이터_전체 스테이션' / 'sampling_plan1_outputs',
}

canonical_train_path = canonical_dir_map[dataset_name] / 'ddri_prediction_canonical_train_2023_2024.csv'
canonical_test_path = canonical_dir_map[dataset_name] / 'ddri_prediction_canonical_test_2025.csv'
sampled_output_dir = sampled_output_dir_map[dataset_name]
sampled_output_dir


PosixPath('/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소_예측데이터_15개/sampling_plan1_outputs')

## 3. 정본에서 2023 train만 로드


In [16]:
train_df = pd.read_csv(canonical_train_path)
train_df['date'] = pd.to_datetime(train_df['date'])
train_2023 = train_df[train_df['date'].dt.year == 2023].copy()

print('rows:', len(train_2023))
print('stations:', train_2023['station_id'].nunique())
print('date range:', train_2023['date'].min(), '~', train_2023['date'].max())


rows: 131400
stations: 15
date range: 2023-01-01 00:00:00 ~ 2023-12-31 00:00:00


## 4. 일단위 24시간 프로파일 생성


In [17]:
daily_profile = (
    train_2023
    .pivot_table(
        index=['station_id', 'date', 'weekday'],
        columns='hour',
        values=target_col,
        aggfunc='mean',
    )
    .reset_index()
)

daily_profile.columns = [f'hour_{c}' if isinstance(c, (int, np.integer)) else c for c in daily_profile.columns]
profile_cols = [c for c in daily_profile.columns if c.startswith('hour_')]

print('daily profiles:', len(daily_profile))
daily_profile.head()


daily profiles: 5475


,station_id,date,weekday,hour_0,hour_1,hour_2,hour_3,hour_4,hour_5,hour_6,...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,2312,2023-01-01,6,0.000000,0.132075,-0.113208,0.132075,0.037736,0.075472,-0.075472,...,1.169811,-0.962264,-0.075472,-0.094340,-0.037736,0.056604,0.037736,1.113208,-1.056604,0.207547
1,2312,2023-01-02,0,-0.096154,0.096154,0.096154,-0.038462,0.057692,-0.019231,-0.250000,...,0.096154,-0.384615,1.153846,-1.403846,1.057692,-0.846154,1.307692,-1.000000,0.153846,0.192308
2,2312,2023-01-03,1,-0.230769,0.019231,0.057692,0.038462,0.153846,-0.019231,-0.211538,...,-0.846154,-0.288462,0.173077,-0.826923,0.076923,0.365385,0.403846,-0.153846,0.134615,0.019231
3,2312,2023-01-04,2,-0.096154,0.019231,0.173077,-0.096154,0.173077,0.057692,-0.326923,...,-1.000000,-0.423077,1.307692,-1.711538,-0.038462,0.653846,0.153846,0.057692,0.057692,0.096154
4,2312,2023-01-05,3,-0.365385,0.230769,0.096154,0.096154,0.115385,-0.096154,-0.134615,...,-1.153846,-0.250000,1.211538,-1.711538,1.307692,0.134615,1.288462,-1.769231,-0.134615,0.115385


## 5. 대표일 선택 함수

이 함수는 같은 `station_id`, 같은 `weekday` 안에서 대표일을 고른다.

핵심 원칙:
- 그룹이 너무 작으면 축소하지 않는다.
- 목표 유지 비율만큼 대표일 수를 먼저 정한다.
- `KMeans`로 비슷한 일들을 묶고, 각 군집 중심에 가장 가까운 날짜를 대표일로 남긴다.
- 예외적으로 선택 수가 부족하면 아직 선택되지 않은 날짜 중 중심과 가까운 날짜를 추가 보충한다.


In [18]:
def select_representative_days(group: pd.DataFrame, retain_ratio: float, min_days_per_group: int) -> pd.DataFrame:
    group = group.copy().reset_index(drop=True)

    # 그룹 크기가 너무 작으면 축소하지 않고 그대로 유지한다.
    if len(group) < min_days_per_group:
        group['selected'] = 1
        group['cluster_id'] = np.arange(len(group))
        group['distance_to_center'] = 0.0
        group['keep_count'] = len(group)
        return group

    # 목표 유지 비율에 따라 대표일 개수를 정한다.
    keep_count = max(1, math.ceil(len(group) * retain_ratio))
    keep_count = min(keep_count, len(group))

    feature = group[profile_cols].fillna(0.0).to_numpy(dtype=float)
    scaled = StandardScaler().fit_transform(feature)

    # MiniBatchKMeans는 빈 클러스터가 생겨 실제 대표일 수가 줄 수 있으므로,
    # 샘플링 1안에서는 안정적인 KMeans를 사용해 keep_count를 최대한 그대로 맞춘다.
    model = KMeans(n_clusters=keep_count, random_state=42, n_init=10)
    labels = model.fit_predict(scaled)
    centers = model.cluster_centers_

    group['selected'] = 0
    group['cluster_id'] = labels
    group['distance_to_center'] = np.nan
    group['keep_count'] = keep_count

    selected_count = 0
    for cluster_id in range(keep_count):
        idx = np.where(labels == cluster_id)[0]
        if len(idx) == 0:
            continue
        cluster_scaled = scaled[idx]
        center = centers[cluster_id]
        dist = np.linalg.norm(cluster_scaled - center, axis=1)
        best_local = idx[np.argmin(dist)]
        group.loc[idx, 'distance_to_center'] = dist
        group.loc[best_local, 'selected'] = 1
        selected_count += 1

    # 혹시 예외적으로 선택 수가 부족하면,
    # 중심에 상대적으로 가까운 날짜를 추가로 골라 keep_count를 맞춘다.
    if selected_count < keep_count:
        extra_needed = keep_count - selected_count
        extra_candidates = group.loc[group['selected'] == 0].copy()
        extra_candidates['distance_fill'] = extra_candidates['distance_to_center'].fillna(np.inf)
        fill_idx = extra_candidates.nsmallest(extra_needed, 'distance_fill').index
        group.loc[fill_idx, 'selected'] = 1

    return group


## 6. 샘플링 1안 후보 계산

여기서는 `groupby.apply` 대신 그룹별 결과를 `concat`으로 묶는다.

이렇게 두는 이유:
- `station_id`, `date`, `weekday` 같은 키 컬럼이 중간에 사라지지 않게 하기 위해서다.
- 이후 대표일 목록을 원래 시간행과 다시 조인할 때 안정적으로 쓰기 위해서다.


In [19]:
sampled_daily = pd.concat(
    [
        select_representative_days(g, retain_ratio=retain_ratio, min_days_per_group=min_days_per_group)
        for _, g in daily_profile.groupby(['station_id', 'weekday'], sort=False)
    ],
    ignore_index=True,
)

selected_days = sampled_daily[sampled_daily['selected'] == 1].copy()
sampled_train = train_2023.merge(selected_days[['station_id', 'date']], on=['station_id', 'date'], how='inner')

summary = pd.DataFrame([
    {
        'dataset': dataset_name,
        'target': target_col,
        'train_rows_before': int(len(train_2023)),
        'train_rows_after': int(len(sampled_train)),
        'removed_train_rows': int(len(train_2023) - len(sampled_train)),
        'daily_profiles_before': int(len(daily_profile)),
        'daily_profiles_after': int(len(selected_days)),
        'removed_profiles': int(len(daily_profile) - len(selected_days)),
        'retain_ratio_target': retain_ratio,
        'retain_ratio_actual': float(len(selected_days) / len(daily_profile)),
    }
])

summary


/opt/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/base.py:1336: ConvergenceWarning: Number of distinct clusters (33) found smaller than n_clusters (38). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


,dataset,target,train_rows_before,train_rows_after,removed_train_rows,daily_profiles_before,daily_profiles_after,removed_profiles,retain_ratio_target,retain_ratio_actual
0,rep15,bike_change_deseasonalized,131400,93600,37800,5475,3900,1575,0.7,0.712329


## 7. 행 수 비교 셀

샘플링 1안을 적용하면 실제 train 행 수가 얼마나 줄어드는지 별도 셀에서 확인한다.


In [20]:
row_count_summary = pd.DataFrame([
    {
        'dataset': dataset_name,
        'target': target_col,
        'train_rows_before': int(len(train_2023)),
        'train_rows_after': int(len(sampled_train)),
        'removed_train_rows': int(len(train_2023) - len(sampled_train)),
        'train_row_retain_ratio': float(len(sampled_train) / len(train_2023)),
        'days_before': int(len(daily_profile)),
        'days_after': int(len(selected_days)),
        'day_retain_ratio': float(len(selected_days) / len(daily_profile)),
    }
])
row_count_summary


,dataset,target,train_rows_before,train_rows_after,removed_train_rows,train_row_retain_ratio,days_before,days_after,day_retain_ratio
0,rep15,bike_change_deseasonalized,131400,93600,37800,0.712329,5475,3900,0.712329


## 8. 대표일 후보 확인


In [21]:
selected_days[['station_id', 'date', 'weekday', 'cluster_id', 'distance_to_center']].head(20)


,station_id,date,weekday,cluster_id,distance_to_center
2,2312,2023-01-15,6,22,0.000000
4,2312,2023-01-29,6,25,0.000000
5,2312,2023-02-05,6,17,0.788710
6,2312,2023-02-12,6,1,0.844587
7,2312,2023-02-19,6,5,1.750814
8,2312,2023-02-26,6,2,1.542182
9,2312,2023-03-05,6,7,0.000000
10,2312,2023-03-12,6,36,1.818206
11,2312,2023-03-19,6,10,0.000000
12,2312,2023-03-26,6,4,0.000000


## 9. 학습용 모듈 로드

샘플링 후에는 같은 노트북 안에서 바로 학습, 검증, 테스트까지 이어서 본다.


In [22]:
WORK_DIR = ROOT / 'works' / '06_prediction_training'
FEATURE_SCRIPT = WORK_DIR / '05_scripts' / '03_ddri_prediction_modeling_feature_builder.py'
MODEL_SCRIPT = WORK_DIR / '05_scripts' / '04_ddri_prediction_train_eval.py'

feature_spec = importlib.util.spec_from_file_location('feature_builder_sampling_plan1', FEATURE_SCRIPT)
feature_builder = importlib.util.module_from_spec(feature_spec)
sys.modules[feature_spec.name] = feature_builder
feature_spec.loader.exec_module(feature_builder)

train_spec = importlib.util.spec_from_file_location('train_eval_sampling_plan1', MODEL_SCRIPT)
train_eval = importlib.util.module_from_spec(train_spec)
sys.modules[train_spec.name] = train_eval
train_spec.loader.exec_module(train_eval)

pd.DataFrame({'candidate_model': [name for name, _ in train_eval.candidate_models()]})


,candidate_model
0,LightGBM_RMSE
1,CatBoost_RMSE
2,StackingRegressor
3,SoftVotingRegressor


## 10. 검증셋과 테스트셋 준비

샘플링은 `2023 train`에만 적용하고, `2024 valid`, `2025 test`는 원래 정본 분할을 그대로 쓴다.


In [23]:
valid_2024 = train_df[train_df['date'].dt.year == 2024].copy()
test_df = pd.read_csv(canonical_test_path)
test_df['date'] = pd.to_datetime(test_df['date'])
test_2025 = test_df.copy()

train_model = feature_builder.add_missing_flags_and_fill(sampled_train)
valid_model = feature_builder.add_missing_flags_and_fill(valid_2024)
test_model = feature_builder.add_missing_flags_and_fill(test_2025)

pd.DataFrame([
    {'frame': 'train_model_sampled', 'rows': len(train_model), 'target_null_rows': int(train_model[target_col].isna().sum())},
    {'frame': 'valid_model', 'rows': len(valid_model), 'target_null_rows': int(valid_model[target_col].isna().sum())},
    {'frame': 'test_model', 'rows': len(test_model), 'target_null_rows': int(test_model[target_col].isna().sum())},
])


,frame,rows,target_null_rows
0,train_model_sampled,93600,0
1,valid_model,131760,0
2,test_model,131400,0


## 11. 샘플링 전용 modeling_data 저장

무샘플링 산출물을 덮어쓰지 않도록 샘플링 1안 전용 폴더에 별도로 저장한다.


In [24]:
sampled_modeling_dir = sampled_output_dir / "modeling_data"
sampled_training_dir = sampled_output_dir / "training_runs"
sampled_modeling_dir.mkdir(parents=True, exist_ok=True)
sampled_training_dir.mkdir(parents=True, exist_ok=True)

train_model_save = train_model.copy()
valid_model_save = valid_model.copy()
test_model_save = test_model.copy()
for frame in [train_model_save, valid_model_save, test_model_save]:
    frame['date'] = pd.to_datetime(frame['date']).dt.strftime('%Y-%m-%d')

sampled_train_model_path = sampled_modeling_dir / "ddri_prediction_modeling_train_2023_sampled_plan1.csv"
sampled_valid_model_path = sampled_modeling_dir / "ddri_prediction_modeling_valid_2024.csv"
sampled_test_model_path = sampled_modeling_dir / "ddri_prediction_modeling_test_2025.csv"
sampled_meta_path = sampled_modeling_dir / "ddri_prediction_modeling_meta_plan1.json"

train_model_save.to_csv(sampled_train_model_path, index=False, encoding='utf-8-sig')
valid_model_save.to_csv(sampled_valid_model_path, index=False, encoding='utf-8-sig')
test_model_save.to_csv(sampled_test_model_path, index=False, encoding='utf-8-sig')

sampled_modeling_meta = {
    'dataset': dataset_name,
    'sampling_plan': 'plan1_day_profile',
    'target': target_col,
    'retain_ratio': retain_ratio,
    'min_days_per_group': min_days_per_group,
    'train_rows_before': int(len(train_2023)),
    'train_rows_after': int(len(train_model)),
    'valid_rows': int(len(valid_model)),
    'test_rows': int(len(test_model)),
}
sampled_meta_path.write_text(json.dumps(sampled_modeling_meta, ensure_ascii=False, indent=2), encoding='utf-8')

pd.DataFrame([
    {'name': 'sampled_train_model', 'path': str(sampled_train_model_path)},
    {'name': 'valid_model', 'path': str(sampled_valid_model_path)},
    {'name': 'test_model', 'path': str(sampled_test_model_path)},
])


,name,path
0,sampled_train_model,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...
1,valid_model,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...
2,test_model,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...


## 12. 샘플링 후 학습 입력 확정


In [25]:
train_fit = train_model[train_model[target_col].notna()].copy()
valid_fit = valid_model[valid_model[target_col].notna()].copy()
test_fit = test_model[test_model[target_col].notna()].copy()

feature_cols = train_eval.build_feature_cols(train_fit)
X_train = train_fit[feature_cols]
y_train = train_fit[target_col]
X_valid = valid_fit[feature_cols]
y_valid = valid_fit[target_col]
X_test = test_fit[feature_cols]
y_test = test_fit[target_col]

pd.DataFrame([
    {'frame': 'train_fit_sampled', 'rows_used': len(train_fit), 'feature_count': len(feature_cols)},
    {'frame': 'valid_fit', 'rows_used': len(valid_fit), 'feature_count': len(feature_cols)},
    {'frame': 'test_fit', 'rows_used': len(test_fit), 'feature_count': len(feature_cols)},
])


,frame,rows_used,feature_count
0,train_fit_sampled,93600,31
1,valid_fit,131760,31
2,test_fit,131400,31


## 13. 2024 validation 모델 비교

샘플링된 `2023 train`으로 후보 모델을 다시 모두 비교한다.


In [26]:
selection_rows = []
for name, model in train_eval.candidate_models():
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    selection_rows.append({'model': name, **train_eval.metrics(y_valid, pred)})

selection_df = pd.DataFrame(selection_rows).sort_values(['rmse', 'mae', 'r2'], ascending=[True, True, False]).reset_index(drop=True)
selection_df


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001427 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2076
[LightGBM] [Info] Number of data points in the train set: 93600, number of used features: 22
[LightGBM] [Info] Start training from score -0.000195
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001183 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2076
[LightGBM] [Info] Number of data points in the train set: 93600, number of used features: 22
[LightGBM] [Info] Start training from score -0.000195
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001034 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not eno

,model,rmse,mae,r2
0,LightGBM_RMSE,0.373824,0.143831,0.925288
1,StackingRegressor,0.377223,0.146351,0.923924
2,SoftVotingRegressor,0.384736,0.151236,0.920863
3,CatBoost_RMSE,0.400230,0.157994,0.914361


## 14. 우세 모델 선택


In [27]:
best_name = selection_df.iloc[0]['model']
best_name


'LightGBM_RMSE'

## 15. 2023+2024 재학습 후 2025 테스트


In [28]:
final_X = pd.concat([X_train, X_valid], ignore_index=True)
final_y = pd.concat([y_train, y_valid], ignore_index=True)
final_model = dict(train_eval.candidate_models())[best_name]
final_model.fit(final_X, final_y)
test_pred = final_model.predict(X_test)
test_metrics = train_eval.metrics(y_test, test_pred)
pd.DataFrame([{'model': best_name, **test_metrics}])


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002584 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2137
[LightGBM] [Info] Number of data points in the train set: 225360, number of used features: 22
[LightGBM] [Info] Start training from score -0.000120


,model,rmse,mae,r2
0,LightGBM_RMSE,0.307139,0.121428,0.941341


## 16. 샘플링 1안 결과 저장

validation 비교표, test 지표, test 예측값을 샘플링 전용 결과 폴더에 저장한다.


In [29]:
selection_path = sampled_training_dir / f'ddri_{dataset_name}_{target_col}_selection_metrics_plan1.csv'
test_metrics_path = sampled_training_dir / f'ddri_{dataset_name}_{target_col}_test_metrics_plan1.csv'
pred_path = sampled_training_dir / f'ddri_{dataset_name}_{target_col}_test_predictions_plan1.csv'
meta_path = sampled_training_dir / f'ddri_{dataset_name}_{target_col}_training_meta_plan1.json'

selection_df.to_csv(selection_path, index=False, encoding='utf-8-sig')
pd.DataFrame([{'model': best_name, **test_metrics}]).to_csv(test_metrics_path, index=False, encoding='utf-8-sig')
pred_df = test_fit[['station_id', 'date', 'hour', target_col]].copy()
pred_df['prediction'] = test_pred
pred_df.to_csv(pred_path, index=False, encoding='utf-8-sig')

run_meta = {
    'dataset': dataset_name,
    'sampling_plan': 'plan1_day_profile',
    'target': target_col,
    'retain_ratio': retain_ratio,
    'min_days_per_group': min_days_per_group,
    'train_rows_before': int(len(train_2023)),
    'train_rows_after': int(len(train_fit)),
    'valid_rows_used': int(len(valid_fit)),
    'test_rows_used': int(len(test_fit)),
    'best_model': best_name,
    'selection_output': str(selection_path),
    'test_metrics_output': str(test_metrics_path),
    'test_predictions_output': str(pred_path),
}
meta_path.write_text(json.dumps(run_meta, ensure_ascii=False, indent=2), encoding='utf-8')

pd.DataFrame([
    {'name': 'selection_metrics', 'path': str(selection_path)},
    {'name': 'test_metrics', 'path': str(test_metrics_path)},
    {'name': 'test_predictions', 'path': str(pred_path)},
    {'name': 'run_meta', 'path': str(meta_path)},
])


,name,path
0,selection_metrics,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...
1,test_metrics,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...
2,test_predictions,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...
3,run_meta,/Users/cheng80/Desktop/ddri_work/3조 공유폴더/대표대여소...
